<a href="https://colab.research.google.com/github/M-Abbi/Probability-Statistics-Bootcamp/blob/main/JPDF_Practical_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Joint Probability practical Example: Pricing an FX Digital Double No-Touch (DNT) Option with a joint barrier.

## 1. The Front Desk Scenario

Imagine you are a Market Maker on the FX Exotics desk. A corporate client wants to hedge or speculate on the stability of the Eurozone and Swiss economies. They want a payout in 1 year ($T=1$), but **only** if both the EUR/USD exchange rate ($X_t$) and the CHF/USD exchange rate ($Y_t$) remain within specific, stable bounds.

If either currency spikes or crashes past their respective barriers, the option knocks out and expires worthless.

To price this and manage your delta/vega risk, you cannot look at EUR and CHF in isolation. Because they are heavily tied to the same European macroeconomic factors, they move together. You **must** model their continuous joint behavior using a **Joint Probability Density Function (JPDF)**.

---

## 2. Setting Up the Math

To keep our toy example analytically meaningful, we will assume the log-returns of both exchange rates follow a **Bivariate Normal Distribution** over the period $T$.

Let:
* $x = \ln(X_T / X_0)$ be the log-return of EUR/USD.
* $y = \ln(Y_T / Y_0)$ be the log-return of CHF/USD.

From standard Black-Scholes extensions, the individual log-returns are normally distributed:

$$x \sim N(\mu_x T, \sigma_x^2 T) \quad \text{and} \quad y \sim N(\mu_y T, \sigma_y^2 T)$$

Where $\mu_x, \mu_y$ are the risk-neutral drifts (domestic interest rate minus foreign interest rate minus variance adjustment), and $\sigma_x, \sigma_y$ are the respective volatilities.

### The Joint Probability Density Function (JPDF)

Because EUR and CHF are correlated, we introduce the correlation coefficient $\rho \in [-1,1]$. The joint distribution of $x$ and $y$ is a Bivariate Normal distribution.

For simplicity of notation, let's look at the standardized JPDF where the variables are scaled to the horizon $T$. The JPDF, denoted as $f(x,y)$, is given by:

$$f(x,y) = \frac{1}{2\pi \sigma_x \sigma_y T \sqrt{1-\rho^2}} \exp\left( -\frac{1}{2(1-\rho^2)} \left[ \frac{(x-\mu_x T)^2}{\sigma_x^2 T} - 2\rho \frac{(x-\mu_x T)(y-\mu_y T)}{\sigma_x \sigma_y T} + \frac{(y-\mu_y T)^2}{\sigma_y^2 T} \right] \right)$$

> **Front Desk Insight:** Notice the cross-term $-2\rho \frac{(x-\mu_x T)(y-\mu_y T)}{\sigma_x \sigma_y T}$. If $\rho > 0$ (which it heavily is for EUR and CHF), a simultaneous positive move in both currencies prevents the exponent from becoming too negative, meaning the joint probability of **both** going up together is high.

---

## 3. The Front Desk Concrete Use Case: Pricing the Structure

The contract specifies that at $T=1$, the desk pays out exactly **\$1,000,000** if:
* EUR/USD log-return stays between a lower barrier $L_x$ and an upper barrier $U_x$.
* CHF/USD log-return stays between a lower barrier $L_y$ and an upper barrier $U_y$.

Under the risk-neutral pricing measure, the fair value $V_0$ of this derivative is the discounted expected value of the payout. Since it's a digital payout ($1$ if condition met, $0$ otherwise), the expectation is simply the **joint probability** that both conditions are simultaneously satisfied.

$$V_0 = e^{-rT} \cdot \mathbb{E}[\text{Payout}] = e^{-rT} \cdot P(L_x \le x \le U_x \text{ AND } L_y \le y \le U_y)$$

To calculate this probability, the front desk integrates the JPDF over the strictly defined 2D survival region:

$$V_0 = e^{-rT} \int_{L_y}^{U_y} \int_{L_x}^{U_x} f(x,y) \, dx \, dy$$

### Solving the Integral

An exotic desk can't sit around waiting for slow numerical simulations (like Monte Carlo) if they want to quote a fast price to a client. They will typically transform this into a standardized bivariate normal cumulative distribution function ($\Phi_2$).

By substituting $z_x = \frac{x-\mu_x T}{\sigma_x \sqrt{T}}$ and $z_y = \frac{y-\mu_y T}{\sigma_y \sqrt{T}}$, the limits change to standardized barriers ($a_x, b_x$ and $a_y, b_y$). The probability evaluates to a combination of joint CDF evaluations:

$$P = \Phi_2(b_x, b_y; \rho) - \Phi_2(a_x, b_y; \rho) - \Phi_2(b_x, a_y; \rho) + \Phi_2(a_x, a_y; \rho)$$

Where $\Phi_2(h, k; \rho)$ is the standard bivariate normal CDF integrated from $-\infty$ to $h$ and $-\infty$ to $k$.

---

## 4. Why This Matters for Managing Risk (Greeks)

The JPDF isn't just used for the initial price quote. The exotic desk's true job is **hedging the risk** after the trade is on the books.

Let's look at **Correlation Risk ($\chi$, or "Chi")**, which is the derivative of the option price with respect to the correlation coefficient: $\frac{\partial V_0}{\partial \rho}$.

By applying Leibniz's integral rule directly to our JPDF equation, we can see how a shifting correlation changes our book's value. In a bivariate normal setup, a famous identity by formatting genius Abraham de Moivre (and later generalized by Plackett) shows that:

$$\frac{\partial f(x,y;\rho)}{\partial \rho} = \frac{\partial^2 f(x,y;\rho)}{\partial x \, \partial y}$$

If we differentiate our pricing integral with respect to $\rho$:

$$\frac{\partial V_0}{\partial \rho} = e^{-rT} \int_{L_y}^{U_y} \int_{L_x}^{U_x} \frac{\partial^2 f(x,y)}{\partial x \, \partial y} \, dx \, dy$$

By the Fundamental Theorem of Calculus, this double integral over the derivatives simplifies beautifully to evaluating the JPDF exactly at the four corners of the barrier box!

$$\frac{\partial V_0}{\partial \rho} = e^{-rT} \Big[ f(U_x, U_y) - f(L_x, U_y) - f(U_x, L_y) + f(L_x, L_y) \Big]$$

### The Practical Takeaway for the Trader

Look at that result! The front desk now knows **exactly** their exposure to a correlation regime shift just by plugging the barrier levels into the **Joint Probability Density Function**.

* If the JPDF values at the co-breaking corners ($f(U_x, U_y)$ and $f(L_x, L_y)$) are high, a rising correlation increases the value of your option.
* If the cross-breaking corners dominate, a rising correlation hurts you.

The exotic trader will use this exact mathematical density profile to go out into the market and buy or sell liquid vanilla options on the cross-rate (EUR/CHF) to neutralize their correlation risk.

# Python Implementation

This code calculates the **Fair Value** of the Double No-Touch (DNT) option and analytically derives the **Correlation Risk** ($\chi$) by evaluating the JPDF at the four corners of the barrier box.

In [2]:
import numpy as np
from scipy.stats import multivariate_normal

def price_and_risk_joint_dnt(X0, Y0, Lx_spot, Ux_spot, Ly_spot, Uy_spot,
                             r, qx, qy, sigma_x, sigma_y, rho, T, payout=1_000_000):
    """
    Prices a 2D Digital Double No-Touch Option using the Joint Probability Density Function
    and calculates the exact analytical Correlation Risk (Chi).

    Parameters:
    X0, Y0         : Initial spot FX rates (e.g., EUR/USD and CHF/USD)
    Lx_spot, Ux_spot: Lower and Upper barriers for asset X (Spot levels)
    Ly_spot, Uy_spot: Lower and Upper barriers for asset Y (Spot levels)
    r              : Domestic risk-free interest rate (USD)
    qx, qy         : Foreign interest rates (EUR and CHF yield)
    sigma_x, sigma_y: Volatilities of asset X and Y
    rho            : Correlation between asset X and Y log-returns
    T              : Time to maturity (in years)
    payout         : Notional payout if barriers are not breached
    """

    # 1. Transform spot barriers into log-return space (x = ln(XT/X0))
    Lx = np.log(Lx_spot / X0)
    Ux = np.log(Ux_spot / X0)
    Ly = np.log(Ly_spot / Y0)
    Uy = np.log(Uy_spot / Y0)

    # 2. Calculate Risk-Neutral Drifts (Mu) for log-returns
    mu_x = (r - qx - 0.5 * sigma_x**2)
    mu_y = (r - qy - 0.5 * sigma_y**2)

    # 3. Define Mean vector and Covariance matrix for the horizon T
    mean = np.array([mu_x * T, mu_y * T])
    cov = np.array([
        [sigma_x**2 * T,           rho * sigma_x * sigma_y * T],
        [rho * sigma_x * sigma_y * T, sigma_y**2 * T]
    ])

    # Initialize the Bivariate Normal distribution model
    bivariate_dist = multivariate_normal(mean=mean, cov=cov)

    # 4. Calculate Option Price via Bivariate CDF combinations
    # Phi_2(Ux, Uy) - Phi_2(Lx, Uy) - Phi_2(Ux, Ly) + Phi_2(Lx, Ly)
    def phi_2(upper_x, upper_y):
        # multivariate_normal.cdf integrates from -inf to upper limits
        return bivariate_dist.cdf(np.array([upper_x, upper_y]))

    joint_survival_prob = (
        phi_2(Ux, Uy) - phi_2(Lx, Uy) - phi_2(Ux, Ly) + phi_2(Lx, Ly)
    )

    discount_factor = np.exp(-r * T)
    fair_value = discount_factor * joint_survival_prob * payout

    # 5. Calculate Analytical Correlation Risk (Chi) using Plackett's Identity
    # Chi = e^(-rT) * [ f(Ux, Uy) - f(Lx, Uy) - f(Ux, Ly) + f(Lx, Ly) ]
    f_Ux_Uy = bivariate_dist.pdf(np.array([Ux, Uy]))
    f_Lx_Uy = bivariate_dist.pdf(np.array([Lx, Uy]))
    f_Ux_Ly = bivariate_dist.pdf(np.array([Ux, Ly]))
    f_Lx_Ly = bivariate_dist.pdf(np.array([Lx, Ly]))

    # Note: Plackett's identity dictates this precise combination of the JPDF at the boundary vertices
    analytical_chi = discount_factor * (f_Ux_Uy - f_Lx_Uy - f_Ux_Ly + f_Lx_Ly) * payout

    return {
        "Fair Value ($)": fair_value,
        "Survival Probability (%)": joint_survival_prob * 100,
        "Analytical Chi (Per 0.01 dRho)": analytical_chi * 0.01
    }

# --- FRONT DESK TEST VALIDATION ---
if __name__ == "__main__":
    # Market & Contract inputs
    X0, Y0 = 1.1000, 0.9200          # Current Spot EUR/USD and CHF/USD
    r = 0.0400                       # USD Risk-free rate (4%)
    qx, qy = 0.0300, 0.0150          # EUR rate (3%), CHF rate (1.5%)
    sigma_x, sigma_y = 0.10, 0.08    # Annualized Volatilities (10% and 8%)
    rho = 0.75                       # Strong positive historical correlation
    T = 1.0                          # 1 Year expiry

    # Client's stable range boundaries (Spot levels)
    Lx_spot, Ux_spot = 1.0200, 1.1800  # EUR/USD bounds
    Ly_spot, Uy_spot = 0.8600, 0.9800  # CHF/USD bounds

    # Run the Desk Quant model
    results = price_and_risk_joint_dnt(
        X0, Y0, Lx_spot, Ux_spot, Ly_spot, Uy_spot, r, qx, qy, sigma_x, sigma_y, rho, T
    )

    print("==================================================")
    print("       FX EXOTICS DESK - DNT VALUATION ENGINE     ")
    print("==================================================")
    for key, value in results.items():
        print(f"{key:<30}: {value:,.4f}")
    print("==================================================")

       FX EXOTICS DESK - DNT VALUATION ENGINE     
Fair Value ($)                : 362,518.2086
Survival Probability (%)      : 37.7313
Analytical Chi (Per 0.01 dRho): 326,344.2473
